# 4. Varredura: espessura e permissividade do polímero

Uma **varredura paramétrica** roda a mesma simulação para vários valores de um parâmetro.
Com ela, você faz a primeira **curva de calibração simulada**: ressonância em função da espessura do revestimento.

Cada simulação demora de 10 s a 1 min. A célula de varredura roda várias simulações seguidas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import emslab

antena = emslab.patch_retangular(f_ghz=3.5)
GAP = 3          # mm
TAND = 0.01


def simular_amostra(espessura, er, nome):
    sim = emslab.Simulacao(f_min=2, f_max=5)
    sim.antena(antena)
    sim.amostra(espessura=espessura, er=er, tand=TAND, gap=GAP)
    return sim.rodar(nome)

## 4.1 Espessura de 5 a 20 mm (er = 4)

In [ ]:
espessuras = [5, 8, 11, 14, 17, 20]
por_espessura = [simular_amostra(t, 4.0, f"espessura_{t}mm") for t in espessuras]

In [ ]:
for r, t in zip(por_espessura, espessuras):
    r.nome = f"t = {t} mm"
emslab.grafico_s11(*por_espessura, titulo="S11 para várias espessuras de polímero (er = 4)")

f0 = np.array([r.f_ressonancia for r in por_espessura])
s11 = np.array([r.s11_minimo for r in por_espessura])

fig, ax = plt.subplots(1, 2, figsize=(12, 4), tight_layout=True)
ax[0].plot(espessuras, f0, "o-"); ax[0].set_xlabel("Espessura do polímero (mm)"); ax[0].set_ylabel("f0 (GHz)")
ax[0].set_title("Curva de calibração simulada"); ax[0].grid(True)
ax[1].plot(espessuras, s11, "s-"); ax[1].set_xlabel("Espessura do polímero (mm)"); ax[1].set_ylabel("|S11| mínimo (dB)")
ax[1].grid(True)
fig.savefig("04_calibracao_espessura.png", dpi=150)

np.savetxt("04_calibracao_espessura.csv", np.column_stack([espessuras, f0, s11]), delimiter=";",
           header="espessura_mm;f0_GHz;s11_min_dB", comments="", fmt="%.5g")
print("Dados gravados em 04_calibracao_espessura.csv (abre no Excel ou no LibreOffice)")

## 4.2 Permissividade de 3 a 5 (espessura = 10 mm)

In [ ]:
permissividades = [3.0, 3.5, 4.0, 4.5, 5.0]
por_er = [simular_amostra(10, er, f"er_{er}") for er in permissividades]

f0_er = [r.f_ressonancia for r in por_er]
fig, ax = plt.subplots(figsize=(6, 4), tight_layout=True)
ax.plot(permissividades, f0_er, "o-")
ax.set_xlabel("Permissividade relativa do polímero"); ax.set_ylabel("f0 (GHz)"); ax.grid(True)
ax.set_title("Ressonância em função de er (t = 10 mm)")
fig.savefig("04_calibracao_permissividade.png", dpi=150)

## 4.3 Perguntas

1. A sensibilidade é a inclinação da curva (MHz por mm). Ela é igual em toda a faixa de 5 a 20 mm?
2. Uma mudança de espessura e uma mudança de er podem dar o mesmo deslocamento de f0.
   Como o sistema real pode separar os dois efeitos? (Dica: use mais de uma grandeza: f0, |S11|, fase, fator Q.)
3. O VNA do laboratório tem uma resolução de frequência. Qual a menor variação de espessura que você pode detectar?